<a href="https://colab.research.google.com/github/riyadewanma2025-ctrl/mgnrega-causal-analysis/blob/main/mgnrega_causal_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Evaluating MGNREGA: A Causal Inference Study

### Does MGNREGA increase household consumption?

This notebook investigates the causal impact of India's rural employment program (MGNREGA) on household consumption using modern causal inference and machine learning methods.

## Problem Motivation

A naive comparison suggests that households participating in MGNREGA have lower consumption than non-participants.

This raises an important question:

> Does MGNREGA reduce consumption, or is this pattern driven by selection bias?

We use causal inference tools to answer this.

In [ ]:
#@title ⚙️ Setup Cell
# ⚠️ DO NOT EDIT THIS CELL — it configures grading and submission.
import os, sys, json
IN_COLAB = 'google.colab' in sys.modules or 'COLAB_RELEASE_TAG' in os.environ

if IN_COLAB:
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'otter-grader'])

import base64, zipfile, io, shutil
if os.path.exists('tests'):
    shutil.rmtree('tests')
_b64 = (
    'UEsDBBQAAAAIANiGiVykqaXiJgIAAPoIAAALAAAAdGVzdHMvcTEucHnNVk1v2zAMvedXcL7IGdwg'
    'HrDLMAfoob0URYEiO7mGoVhyIlSWUkleERT976PsfMfBkraH+OKIeqReyEfKD3f57cPj/fUYEhib'
    'mvd6jluHi7ce4BMoWvHgFwQvcRC1lrkWylm0xcOlxdYCndCSNmv/vK1/NYiC2j1AN3DjoJk/1gRB'
    '0AkYjUYgrFDWUVXw8CXOneHUcZZXnKoIQuQYQSk1df0+UMX20IVWzmh5IlpR8ZfnmBdR4Rl7+E5+'
    'TSpPI2655IUTWuVUUbmwwkbAROFaIlTK8BmEgk4klNpAs52Sze5EUEsiIFZMlX+/zhYk+zRPwye1'
    'c1RGYJ1pyW1ZB2gU87AP3xIgeZ6T847DOkfdGzPBGFeohVsqLT8Ckrp45mwFOsC8H7p9QncrqWGT'
    'NBIIWTlAAikrU1JNleFTSjJIEogx9yg0W1dzXxeSDbzcwiOSaUIvdfnf0MOzQ0+0liGd2P1egav1'
    'H+rDb4h/DlvdtcjtPvHI5XqJPFNTJxTZ+15EjZuW/3CFMVVfV7lN3yOfJbErGD/+ucmvxzcnFXx3'
    'gPlCNpaOgh/OmIPJknn/te1jUvAEj53mp1aGyXvlJuyv5wqmjijMokPeZDV9eFmif86EacN0es1q'
    '4+zXz6NLkaqo5to4MMcTLbkKDR+UQjF/nRjyxNInFmXf8XLYGuF49Y0S+HFJidqxZLtBAlto0zgf'
    'HhC4xbz5bGG68N8zm+y1IbPee+8fUEsDBBQAAAAIACm/i1xpt/ktRgIAAPkIAAALAAAAdGVzdHMv'
    'cTIucHnNVj1v4zAM3fMrdF6cAEaGTocCCXBLl8OhQNEtCAxVYlpdZNGV5DZBkf9+opwPu3YuLrJU'
    'gy3RJB9FPhK+/53f3T/8+fXIZuzRVjAaeXA+HD5GLKzE8AKSW5a83iRZLSlRGe+C7Ode4CoVbIJg'
    'Ec+0Po67qCG4+6TQr3gyQEmoNkmSXoX5fM6UU8Z5bgSMX29yyZ9zixpcxqQSfsK4kcyBb32bsFm4'
    'WFpahTYXaFxVlF6hSTOWurXS2uXPXBmQJICi1LgtwPg8wPjKpbveWGLWhgXZcAmbUnPDCT1jzts6'
    '4LM606CiyvGE/ZixNM/z9GuhhDxm/R9elJRgQq7vuHZwRkmjWIM8KHV0dl2zK+oqYcWoOD6ULC+4'
    'Fy/gxpuMhWyA8CAnt72m0+k0vtWqmfRg94SoL9jQsuAra9gmWB+hvgZEVbyM88Z16K7NoaBTje9g'
    'x5OLdgHuENZlkMaFCE+ZwHofyEG83oKj1z7DtOXyb+U87ZQRupJwhuk96Tp6XxEzyIVBekpkBj1r'
    'gMiwbZ5hMwRqD9PPO1rEmGaLL3q6e7lI/baEdEntTxdfYWUk2DT2XIdrA9wd7rHMYs/1V++/g6EN'
    '0h4+7XgLkIp7HBZtx1Ej0pjEq0PtjsXP6dVaDU1ur7MrQx4w68j2O4w6brbj9Ts10PnBv58PbIWW'
    '1bqLU45DH2EJJjb0ExdriRiFrqwCbatarvih4SPva13Z7EUHOgyWyO1vleyWZNl2kjiBNhp3ARLi'
    'I/21SBT0O3OqQO1yOdqN/gFQSwMEFAAAAAgAx1WMXFCJ4sgJAgAArAYAAAsAAAB0ZXN0cy9xMy5w'
    'ec1VTWvjMBC9+1cMvtiBJJceFhYS2D30UpbC0lsIQbXGiRpZUjXyekvpf9+RnLRx4yyUZaG+WPP0'
    '5kMzT/btzeb69uePb3ewgDvfYpYFpMDGcwb85EY0mH+F/PEqn/aIs8oEYuzLAaBWsQ8Dq2TH5/l1'
    'lRiVoHeEceKbg5Uxq8/zfJSwXC5BkTIUhKmwfLzaCOIU1KAJU5CqChMQRoLQutyDMjBgQG09JHhV'
    'aOG36IspFB4FWaPMNhpcccCN8/ZeY5MAb4lqFUaxWv0u1pPRSlNPR8+YHxp6trFTUqLh418LTXiB'
    'pG21R3kknXFezt3+odWD7r32bD2n4JUrJ/PWOfTlBBYLKL4XH+tDjC/MU7nvzua0OpnJeq5tl5Kk'
    '4XX99Ag1VkFZE8dhcCuC+oVxrTkEnAKVNbVtjYzreyUovgOHDyhhZ1vCndWSQHgEZ63vJdGvTvaT'
    'N4tZK5OiOo+zFCWWGwHOgJ5VGdK2EYfsXnTQoDDHtVR1jR5ZuxER8qGl5F5pS5wwWAg7hMA96+ts'
    'Y1Wk4mEA2bUK/0Nv0fczyO2iHIa3clwSiXNp4tJ2phNefrh7fy9r8B0YrypSYgU1yyi+bRtmtp4d'
    'zSgorqvR0SDROI1ATqukih1qOWOHtIdOeJYXmFZR/Ph9MiEMkPUwSE4V3yc5miAPTy79aKSt4h/o'
    'TR19yHX2kv0BUEsDBBQAAAAIAISqilzCwD4KKAMAAIkLAAALAAAAdGVzdHMvcTQucHndVltvmzAU'
    'fs+vsHjB6QgD1mhVtVSa2uVlWyN13RNCyMWmtQo2xc6WrOp/37FpQ26kt02aFkXk2Jzzne9cfJzJ'
    '53Q8Ofv68RyN0Hk9Zb2eZkrD4raH4OMIUjLnEDk3+47X7FSSC61gLxze76gpByPYie3afG4XktXI'
    'iFpT2K7YGkhq3NaO42xVODo6QlxxoTQRGcM3+6ksVJpJluc840xoD2Funnkhie73ERF0i0GV/iDF'
    'lK0pb/Vok9NFJc1/FpAzeKaQPV4SzTDNPeRmUqhpWWkuhQvL8lLU7JKAeDw5HU++n558OkuPJ1++'
    'bfe5FqXx4iHKM93EQ4oCXyMuGve5rJFdxa7hsZQL43me1kxxOiWFMku9tEyeGTDUxNv+4opTygTU'
    'bQy4rEOpkNk1ow9KGzp3m2av6ZGykrVGAmowR0QhUXmogtyBCN+Keghyq1UJUIXyScXNtiq7Cw0F'
    '1TW0jodSKHjKhGGR5ozoKWTUFv3JpU0p1OBSAI4qfUKpATel1riiPsgZ0TimeRwv+iZJfKL0vGK4'
    'aVWvJZR4iMy4GoWweUXUAmvkArK7g0Q+LYqGApCFAOKVpt102JDu+znXeBfsXzsRF1IWmFwoex42'
    'mz1BgyYovyI1KdVS9vroAwr8IHzBCV843Rw1u/0Ngz9/vIzt/3i6ROXXYC5LXzFG8fsd7WVOzX4Q'
    'dCvMQtBoAYWsS1JgxX+xUSp2AM+il9lVYBaitwiH6I2xZ7MKD/Ag8CNYh36A9iynATTggZWjfsdN'
    'Y+H0CosLLmTJgUcIB7DaYTYHsygIAvA5ND97Bgn8D4OWwMG9HDVE1wLNSMFGkFmo3aMxl1xwcAjT'
    '6oRoMobuZ/jWnbuHQMRcMkaAa9WdhUaahUaMrBjdPQK7bX7YF+YuszcYPGKD3IB2XGItXuesta9h'
    'xtprssVLnjlGG9Irs7RBBrqAtUzisdnZDriHTHRMuYXPxejRrxpy9yRLEwM8n5h5D4k0lwVVo3ce'
    'aropNeeejXYc4LUYwVvsgomNy3SuiSIC4Z8anis7ySqIozJZW+NNB465Qc2/aCoz8/e6nagNZNK7'
    '6/0GUEsDBBQAAAAIAKOFiVwTKE6EBQIAAL0GAAALAAAAdGVzdHMvcTUucHnNVU1v2zAMvftXEL7E'
    'AQwj2dAdhjVA0S2XbTXQdafAEBRJyYTKkmfJwYqi/32UnMaJ43TZ0MN0sfjxSIp8hPPPZJ7ffr26'
    'g0u4qxsRRU5Yh8JjBHhiTUsRv4f450WctprKSO0s6qYXW41tJIJQswiyP4+7W/Bg1PYchh07gOE+'
    'bR3H8aDDbDYDwiVdY6W8VOGqjXWS2YSvUhgxo21TVk4aPUKxXOtarCler/Obef795uOnW3Kdf/k2'
    'PhleWqmto5qJJIRPgUvmxkA1B6pUcg9Sb2tYmRqCuBiZxjFTClK/8Wmr2lRCW+keCG2Y15iNqBWt'
    'yEYaRX11pGLOG6gTBNMtlfCSpnIjCE5ClmgIQGUPZP/qfZnVgsulVCGXxXbbUmg3KoYfGEZ97ssX'
    '+8UVKSRLY1QKusr8hYzbnmxdT9XRtqe1tm9ktHkejzaO7EzF31WMFEmHDT8k50IjjeZUWXHCSRl2'
    'L/iz05HP0zHs/6UswRBc+gAWkxwvmz9ZloXvdlx7fC1gBpPs7XCferAesVvou7OgwxtQwAeYTs4K'
    'cEDGk4BhFoUurahUgmOHsN0JUg9Y2GAWFrpr4Qt9Fr8qwVyI0VEa5KqLfQkTEEgoSHY879mnrf2Q'
    '+y/k/MN6YcRdWa++QB77j/tzoOnNK7bM1AF8nCB2D1X483DD/C+pW6o2ZBE9Rb8BUEsDBBQAAAAI'
    'AHS3i1zuB09BswIAAKsKAAALAAAAdGVzdHMvcTYucHnVVt9r2zAQfs9fIfySBLLQMtaH0gS2NBll'
    'aw1t9hSCUKxzKipLrqSwhdL/fSc5cX7Ups4YjPkh0Z1Onz+d7j45/kYn8f3t5ykZkKlZQavlwDo0'
    'XloEn0ixDKJLEj1fRL3Ck2uhnEXf+aeNx64ELkLPLNj+eSlHISJh9iigOnC3QHP/WhNFUWXAcDgk'
    '1IBFokhXZMwBfQQHRi9BgV5ZCmkKibMdnvZIO9HKrrLcCa3aaGZLZWDJcDiK7ybxj7vr8T0dxd8f'
    'emQ8mYxHU3obX99Mbjbebi0FYYWyjqkEOp5Nj3CRuC5hihMmZeeJCFXQTLUhwZq1E891S9oGOsAU'
    '9W5vhOnFmi6NXuXe4XROF7ipVCSCmTXNjU6FhPa8mlY4xDq+FlwgOjt6zbxLBnjk7RQyJn0mGQce'
    'qB2aD6PwO/W/8ZdgfEVqhkk/NJDpYhMKmKFO/wzZlpgOCdZuxwj1ejr1hdZyw32XrzkZkrOQbglq'
    'f2e7/BZb89M87Z6YMqy+XvXEo+AcFFbohEkLNUFSJ0/At0FvYl7fLvvPuyHLtXFEIfaaMEtUXs/W'
    'nxLSVXmfWWYMW9ecHraUW+cwSKVmrv7VFH7luEHgpXIdP/1+P/wf1fhlQWXG01nbu2gxjZWFVXM+'
    '7/ta63SrD7iEbAx41hAQ+2wfB+UTs+LNpTbrAsrHNEWbNkCbNkXzff8unA9qiLcVkHcxt4ENcTdq'
    'tA/LRSHWKE1BnehTFiTkY9Nz2ena+7BXg+a4pUTuw3onZYnxbUCuGhfjRmJrkYaN6rpaoUshxr4V'
    'NpHaQvV9MjtqM+zjskUr5pjTcnAOHy56BJ5XTFLF1MDL8qmKfQJBX/EHrArHv6BSHv8hoX33X6PV'
    '4E7za//wSjvwzA9BIptoExa/fUHkRd5/Z3Kd+A/Q3T1XQM5br63fUEsDBBQAAAAIAGNfjFyZZ8Id'
    'cAIAAOQJAAALAAAAdGVzdHMvcTcucHm9Vk2L2zAQvedXCF/sQFpoLwuFBnrZSykLZW8hGFkaJ+rK'
    'Gq0kZxOW/e+V7HzYkbJtUqghkJk3oxm9mZH08L28f/j549sj+UoeTQuTiQPrvPA6If7LFG0g+0Ky'
    '57ts1ms0CuWs1336vNfYVngnr1l0cvhej/86C0btmUHa8OSAPIQ1WZYlDebzOaFSFsIKZR1VDAqs'
    'fs0IF8xNSY2GeJEIRRbPd6VVQmtwJZ2RgVSNJLacJgN1nLyXQfg9hUghYAjcCYscjEFTup2GfEZy'
    '2GpJFXUCVRBrsc2X/ydPz+AsDawF56A8y/dUWrhgJJE9AT8YRTZvsds/VLSkpSfKN9+QjMWIvOVH'
    'iS9gijQL+0U8vdEiHeV/dC6KIm+AC+rQ5KEufUpT4itV5Bqt++AMUNeAcmmcvIdTt36huyEwJVRx'
    'HzbnBvUB8LkePAzEAUdwHG8AJ/Md42n/6Q3TUFZx9aqrq1fF1auuqF7HSTgF+t1UQ/Z7IOQRYRI2'
    'IC3BmjBUtm30udmxTow6iP2PJBKoa0hFr0BBLRLAiopEQhUw2loIGZ0VaJ9PMqFTaYM3lbHeHzn+'
    'fI/1vEkYr8GBwZXPHFs7hG9qDxa3B7u6PVjcHuya4cYNGEn3k8bGsyuc2Ai3izHhz0knagE8xnjg'
    'qBFKWCdYDDNsGlTEtlqjcTF+EYCtM1SjpAlMYd8UqXw81qqLqPanDCib3KVC57u/0dTQSkJyKwns'
    '2Ifx0qdGOmP9BNANCh6rbVutDLYJB2dEE2tDT5vj0A+RNP+39PBfXKTB98Z7dKRZjhfJLPNTy5MB'
    'svDCCA80jiy83E6Xa7/kcvI2+Q1QSwMEFAAAAAgAdLeLXMKeYAiNAgAAXAkAAAsAAAB0ZXN0cy9x'
    'OC5wedVW32vbMBB+z19x+MUOGM9JGugGCewH2cMYhbVvxQhVuiSituRKCl0o+d93spO2a5yUbl1h'
    'xpyk03ef7uTTyWff2Ozsx/ePFzCBC7vCXs+j8zS46wE9keYVRh8gujmN0lZTG6W9I90g32rcSpER'
    'aS6bcXju7nsNQnD3BNANfDAwMixroyjqBEynU1BVbawHvarqNXAHuk6h5lpSl95aHjRkck4B1jL7'
    'wj2fWYow6XYjy7KmjZfIJZtjxUuMKYo8hUEKrRwU6XFbit0jE9zjwth1MI+/okbLyziF+PxzkGef'
    'mub8Yqt6jlMq57kWyLyh91az6yoQj1M4CV4NUxhTfzR6jqek7WJcWHTBfJSRUZ6R9SCIPDsJIj9C'
    'sukf3uQQMW2zrjNuLV8nl4M8J+feN3LUyMGwaU5JFilIv65xMi8N90doyVdi9dwu0Cu9oLEwVYX0'
    '2b0yOqFvm7Zrp3C1kgRic8tFmJtQQIeJlVO63dQkrEHuKOH7QBsEJepGdxm3y6JkStM07VrRh8kE'
    'hp2szXE6FgfxGc9LdkXpMFee7djjAroBlrwxVVy8fDmHfhuCxMosLK+XSrDamrmilG6juItr4XdZ'
    'TnkYRk4w53cDS6b+firkTonOxZuXuUNHujuhoqWSEjUd+xkvHR4AlUZco9yB9jCbfbP/v8RQ519U'
    'GZI7zGsVHDrMr1JzUhhm4zcpPDQYv1X5GR8h9vgzXLvtIcUbulLXjDu6Nh2x+7jISnOLNjnMkCTx'
    'Nn1A6ZavD8ZCEt8aonikbasa4Z14R6f7KdyJfVUXaolyVaKEJvWOzHurrh7Pd4fwN/Ui2P5hufhN'
    '8yThIieMbYz3F4hCtoQfI2lE+GN6qCEtZdHb9H4BUEsDBBQAAAAIAKOFiVzopLK+ggEAANQEAAAL'
    'AAAAdGVzdHMvcTkucHnNVNFKwzAUfe9XhLx0hTLEJxUdyHQv6gpTn0YpaXLXhaVJTdLJGPt302xz'
    'zHWKimCe7r05Jzc555LkLhsko4frJ3SFnnQNQWDBWJcsA+QWlqQEfIHwyzmO15VKcWmNq51tCqbm'
    'juMKY583a/keeQQl5gOgHbgjKNZ01RjjVkCv10OZVnntr7oOJBiT0SnQWYdNYhRSJU1dVpYrGbq0'
    'LKSGgriwnwwHyfPw5naU9ZP7x+hoB264NJZICp1Ns3FYCUIhV1mjUpjGqOPEiNFEKGKjCBHJPmVV'
    'xDglPC9XSsRIVt0myKL2W3hHvhBgHKo5aCJERjUwnnPB7SJMEZdoHE55MfWvdzt12URCvYbp95o5'
    'F+L2jSlnDKRzakCEgSMgoegM2BZ0gFkd0v71VJDcHBuHCF2i0+6JH4PG1uMD8Bu7DUjDLZ87m7OS'
    'FJLbmsHWcFO6SfAvc2JpYsF7TnQBf+B6w/2h6XuVdP8QbKjSnnzYANtF5T8kpmgj+m4S1kemwSp4'
    'A1BLAQIUAxQAAAAIANiGiVykqaXiJgIAAPoIAAALAAAAAAAAAAAAAACkgQAAAAB0ZXN0cy9xMS5w'
    'eVBLAQIUAxQAAAAIACm/i1xpt/ktRgIAAPkIAAALAAAAAAAAAAAAAACkgU8CAAB0ZXN0cy9xMi5w'
    'eVBLAQIUAxQAAAAIAMdVjFxQieLICQIAAKwGAAALAAAAAAAAAAAAAACkgb4EAAB0ZXN0cy9xMy5w'
    'eVBLAQIUAxQAAAAIAISqilzCwD4KKAMAAIkLAAALAAAAAAAAAAAAAACkgfAGAAB0ZXN0cy9xNC5w'
    'eVBLAQIUAxQAAAAIAKOFiVwTKE6EBQIAAL0GAAALAAAAAAAAAAAAAACkgUEKAAB0ZXN0cy9xNS5w'
    'eVBLAQIUAxQAAAAIAHS3i1zuB09BswIAAKsKAAALAAAAAAAAAAAAAACkgW8MAAB0ZXN0cy9xNi5w'
    'eVBLAQIUAxQAAAAIAGNfjFyZZ8IdcAIAAOQJAAALAAAAAAAAAAAAAACkgUsPAAB0ZXN0cy9xNy5w'
    'eVBLAQIUAxQAAAAIAHS3i1zCnmAIjQIAAFwJAAALAAAAAAAAAAAAAACkgeQRAAB0ZXN0cy9xOC5w'
    'eVBLAQIUAxQAAAAIAKOFiVzopLK+ggEAANQEAAALAAAAAAAAAAAAAACkgZoUAAB0ZXN0cy9xOS5w'
    'eVBLBQYAAAAACQAJAAECAABFFgAAAAA='
)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(_b64))) as _z:
    _z.extractall('.')
del _b64

import otter
grader = otter.Notebook(tests_dir='tests')

# --- Data loading (compressed for readability) ---
import zlib as _zlib
exec(_zlib.decompress(base64.b64decode(
    'eNqVWFtv2zYUftevILAHS4uiSk7cGcVcYHXirlubDEn6ZBgCI1E2F4nUSDqXFv3vOxSpmy27rWHE'
    '4rmf75xDiqFFyYVCbFuULwhLxEqHGlKJWQoE+JZpTZMKK1nwlOQywCXVTFk4meAFkg85wYIF91gS'
    'ZMWTnDPSZxMmSXGfNyLvBU4pYeod51JRtr4ha0Gk5MLfY81zLCXNKBF9kwVRgiaytijGsUy4ID4S'
    'PInxNjHLHR2dQyxJThJFOat1/17wPPXRrRJYaU9pRTCqJOGsyIO0yGvpj5SBsYtPH300x1uJ8wX4'
    'kQoINV5P4Asil079EGQ0V0TUS3dE1wyURp7jOKwMBGDOi0ASkrrnY89haIZeh2HoaOBJzHBBJJCW'
    'oxv8L5Zqg9nIR6NPON28YPQP4EXkRlPe0Q0W+uEv+H0AsVQvrlMqN1g/zTcbrBSVayw2o5UDdA2h'
    'knFJRFz5Ai8Th8UNB9Y5YW4nDg/9igYUncYaqOiUSEmwciGzNXE7Bj0fMfTqFeqSnNo16GEh8Iu7'
    '7Dhcplp+wOUKZVygFFHWcFcAaAIIkTgBgTUXLzYaA3Cy4TQh7nJ0O9dw3N5V8LyrFu8JIwLnoxUE'
    '6KNytgyD8dhHYRBN9d/zUP8dh+Bhw7eSbKBBYkm/2LCTnJZu66jkFJqZuefBBMx56ARFPgJrERR3'
    'Q3Aa4/WwInRFgXNoApANtWoA2byUxKVMAXQ6ot9CayMjIEp6Cd5TxgsKBqI6cgaIkHQbV/PZFSXP'
    'JUwpU1q6CrIVO6nknjZEEHcXzVmLlI/OwMuP6xmgIw1JV+u0o9XLa2ZBq/MFDbCnx7ZFrrbShyk0'
    'SDs5pHo888gUqCP5w8mPg7CK7YdVTf5hMN1ROw5AaPGq5HEiqp2gTr8xYnKeeIHgW5a6kWfmEbOE'
    'xIrD94nFD8Vgy/XgGNt2HVfIT8OuwZyv41JQmLkZch0Enwgmono4gTDD17Az9AvV8qbAazOo6Kea'
    'Hp7Z7WQn0kYzmoDAsUI0E5LlHCuv7/JQGYaV9sawmvmzqkE8p0o9htNAwqHZtiHA5zbIGJSoMmdP'
    'g9PpOJjYnKHhIC7QAzl33+QrFJGJ1+IzPYLp+DvQvT4AwO38IGiHMIOd8pDK2U/DHFYQdDq9n26n'
    'TTTovIR3B6r0Rh4BPG5kCqVhP+1g7XlOsWaCrPHBDbG1BWVSYmsCboo01QeuiXACjwMhnqBzw/gp'
    'iM6O6ByCaDwxOkMz/BYsDmmdNp46WwUID3uIrIfmNPodnU2GzUZhR7TdhN+i6a68Pn1782GQ1XCG'
    'fQQHe9rGv7NPNLH2z906uEl4dAymh9E/voMcKfWx1tZDtzfWlls36K+o6b4jO080hgjs3uOkGaBZ'
    'psEFVngh4LXI/Vqpjuo3n9Gb5iXIN5zqHQnI1a+l9SEEZp9QS9me0Hz72OWYiaiZZtXlN2WtRRqC'
    'lepDClJ9gpVq2wAk2oXf5r1TcQvBDtXK79UEpPdoVtbWCSTsUx13T3tfr6kqcJtn3/kG9fsFXV3f'
    'XXbIiMId6xHTHOtb0T1J4CZBkNoAGb7yhakNUTRBKRQ8QB8YEgTniDzifFtBKX30wreIkUci0APj'
    'T6BLKvOIZBlcbwLn7ubzZfzH3SV0jjnE02zZiXEFNyjMXNg459dXi+vPVxeXN/H8+uOtvmkMtovf'
    'aQ2/3wz+Xu2Ha+336uoP1/Fgyfy6q52Vc7lYXM7v4k/XFx8WH/Yj78e2H8URv/0A9zJuU4QoYDJJ'
    'hmLCErhZgkOstqDn6qplekx9aJN8W+hyZXBU2cXsCt64fIRTiBHn+T1OHixrtsC5JN6bKhKa1dpm'
    'rT/GU2p2gzVRcbotCtp1ubQ6cJFJ4cSLMyqkmt1B3c0uBZd4csBeu7tQlpLnWWMzqNZeHVUnFd2t'
    'jCukMxqyap/gPliZcGsEOiY0NHke69YmM3gJaNwM4IOgNo1NuNWWZBmt9KYcfjelr6Oql6Dqqi4U'
    'zCnsvF+I4NLV19wmX8/75qMjGAgC6qwJpHcU+CiH3nJrnk3Sq1tF4gw68THOYKykq2CsVQFv3z7Y'
    '/G9LoE1Tw7I90AjY67E0F+SG7LVSVcpwAXLdjhK8NXkBDJCecy0JECjB80HJsCdpcyzwswtXgYIy'
    'dydCv3br11a9Jktd3rz6P0nM89488K1KeEF0Pf02O7O0ZvSizt+SwFkMMR8ftI5ypZsSSdfM9AFw'
    'YWrdZTsmy57z1Wq3irVnmCP8TOUs2jEqi0A3aN1TrmH4aINlQ5yNQGTUwxPUYKfqjGsHkL0YjE0v'
    'ADhhch5jzZyN/pyfjbz/AWy2V2Y='
)).decode())
del _zlib

# --- check_and_submit() function ---
_SUBMIT_URL = 'https://script.google.com/macros/s/AKfycbxuhfgxGk9F5l6WiiNCo146PLo-RuaZfZd5L_eVZzTVjB0pf2nEs4WEiCw-9rezXPUB/exec'
_ASSIGNMENT_ID = 'D'
_QUESTIONS = {
    "q1": 10,
    "q2": 8,
    "q3": 7,
    "q4": 15,
    "q5": 15,
    "q6": 15,
    "q7": 12,
    "q8": 10,
    "q9": 8
}

import requests as _req_lib
_EMAIL_CACHE = ''

def _prompt_email():
    global _EMAIL_CACHE
    if _EMAIL_CACHE:
        return _EMAIL_CACHE
    _email = input('Enter your Ashoka email for submission: ').strip().lower()
    if '@' not in _email:
        print('\nPlease enter a valid email address.')
        return ''
    _EMAIL_CACHE = _email
    return _EMAIL_CACHE

def _submit(scores, total, possible):
    """POST scores to the submission endpoint."""
    global _EMAIL_CACHE
    if not _SUBMIT_URL:
        print('\nEndpoint not configured. Show scores to instructor.')
        return
    if IN_COLAB:
        _email = _EMAIL_CACHE
        try:
            if not _email:
                from google.colab import auth
                auth.authenticate_user()
                import google.auth
                from google.auth.transport.requests import Request as _AuthReq
                _creds, _ = google.auth.default()
                _creds.refresh(_AuthReq())
                _profile = _req_lib.get(
                    'https://www.googleapis.com/oauth2/v1/userinfo?alt=json',
                    headers={'Authorization': f'Bearer {_creds.token}'}, timeout=10
                )
                _profile.raise_for_status()
                _email = _profile.json().get('email', '').strip().lower()
                if _email:
                    _EMAIL_CACHE = _email
                    print(f'  Submitting as: {_email}')
        except Exception as _e:
            print(f'\nCould not verify your Google account automatically: {_e}')
            print('Continuing with manual email entry instead.')
        if not _email:
            _email = _prompt_email()
            if not _email:
                return
        try:
            _resp = _req_lib.post(_SUBMIT_URL, json={
                'email': _email, 'assignment_id': _ASSIGNMENT_ID,
                'scores': scores, 'total_score': total, 'total_possible': possible,
            }, timeout=30)
            if _resp.headers.get('content-type', '').startswith('application/json'):
                _result = _resp.json()
                if _result.get('status') == 'ok':
                    print(f'\n{"="*50}')
                    print(_result.get('message', 'Submitted!'))
                    print(f'{"="*50}')
                else:
                    print(f'\nSubmission note: {_result.get("message", "Unknown response")}')
            else:
                print(f'\nServer returned unexpected response (status {_resp.status_code}).')
                print(f'Response preview: {_resp.text[:200]}')
                print('Your scores may still have been recorded. Contact the instructor.')
        except Exception as _e:
            print(f'\nCould not reach submission server: {_e}')
            print('Your work is saved in Colab. Try again in a minute.')
    else:
        _email = os.environ.get('USER_EMAIL', '').strip().lower() or _EMAIL_CACHE
        if not _email:
            _email = _prompt_email()
            if not _email:
                return
        try:
            _resp = _req_lib.post(_SUBMIT_URL, json={
                'email': _email, 'assignment_id': _ASSIGNMENT_ID,
                'scores': scores, 'total_score': total, 'total_possible': possible,
            }, timeout=30)
            _result = _resp.json()
            if _result.get('status') == 'ok':
                print(f'\n{"="*50}')
                print(_result.get('message', 'Submitted!'))
                print(f'{"="*50}')
            else:
                print(f'\nSubmission note: {_result.get("message", "Unknown response")}')
        except Exception as _e:
            print(f'\nCould not reach server: {_e}')
            print('Your work is saved locally. Try submitting again later.')

def check_and_submit():
    """Check all answers and submit your progress. Call anytime!"""
    scores, total = {}, 0
    possible = sum(_QUESTIONS.values())
    for q, pts in _QUESTIONS.items():
        try:
            r = grader.check(q)
            if 'All test cases passed' in str(r):
                scores[q] = pts
                total += pts
                print(f'  {q}: {pts}/{pts}')
            else:
                scores[q] = 0
                print(f'  {q}: 0/{pts}')
        except Exception:
            scores[q] = 0
            print(f'  {q}: 0/{pts} (error)')
    print(f'\nTotal: {total}/{possible} ({total/possible*100:.0f}%)')
    _submit(scores, total, possible)

print('Setup complete! Call check_and_submit() anytime to check progress.')


## 1. Exploratory Analysis and Naive Estimate

### Naive Comparison: Treated vs Control

In [ ]:
treatment_summary = df.groupby('mgnrega')['consumption'].agg(['mean', 'median', 'count']).round(2)
covariate_summary = df.groupby('mgnrega')[['head_education', 'land_acres', 'distance_to_town_km', 'prior_consumption']].mean().round(2)
print('Consumption by treatment status:')
print(treatment_summary)
print('\nAverage characteristics by treatment status:')
print(covariate_summary)
print(f'\nTrue ATE (synthetic only): Rs {TRUE_ATE:,.0f}')

In [ ]:
q1_treated_mean = df[df['mgnrega']== 1] ['consumption'].mean()
q1_control_mean = df[df['mgnrega'] == 0] ['consumption'].mean()
q1_naive_estimate = q1_treated_mean - q1_control_mean
q1_effect_direction = 'hurts'  # 'helps' or 'hurts'

q1_selection_analysis = {
    'selection_bias': q1_naive_estimate - TRUE_ATE,
    'sign': 'negative',  # 'positive' or 'negative'
    'why': 'Participants are poorer, less educated, and have lowprior consumption than non-participants.',
}

q1_rebuttal = 'The naive estimate of about -11584 is misleading compared to the true ATE of about 10008 because it reflects selection bias, not the true causal effect. Participants are systematically worse off, so comparing them directly understates the programme impact.'

**Insight:**  
The naive estimate suggests that MGNREGA participants have lower consumption. However, this comparison does not account for systematic differences between participants and non-participants.

## 2. Causal Framework

We represent the data-generating process using a Directed Acyclic Graph (DAG) to identify:

- Confounders (e.g., prior consumption)
- Mediators (e.g., skills gained)
- Colliders (e.g., employment status)

This helps determine which variables should be controlled for.

In [ ]:
q2_dag_roles = { 'prior_consumption': {'type': 'confounder', 'control': True},  # use True/False for control
    'skills_gained': {'type': 'mediator', 'control': False},
    'employment_status': {'type': 'collider', 'control': False}}


q2_employment_explanation = "Employment_status is a collider because it is affected by both MGNREGA and innate ability. Controlling for it would open a backdoor path between treatment and unobserved ability, introducing bias in the estimated effect."

---

## 3. Naive vs Causal Estimation Approaches

### Naive Estimate

We begin with a simple comparison of mean consumption between treated and control groups. This approach does not adjust for confounding.

**Code A**
```python
treated = df[df['mgnrega'] == 1]['consumption'].mean()
control = df[df['mgnrega'] == 0]['consumption'].mean()
effect = treated - control
```

### Causal Estimate (Partialling-Out / Double ML)

We now estimate the treatment effect after controlling for observable confounders using a residualization approach. This isolates the variation in treatment unrelated to confounders.

**Code B**
```python
controls = ['head_education', 'land_acres', 'distance_to_town_km', 'prior_consumption']
m_hat = GradientBoostingRegressor(n_estimators=200).fit(df[controls], df['consumption']).predict(df[controls])
e_hat = GradientBoostingClassifier(n_estimators=200).fit(df[controls], df['mgnrega']).predict_proba(df[controls])[:, 1]
y_resid = df['consumption'] - m_hat
t_resid = df['mgnrega'] - e_hat
effect = (y_resid * t_resid).sum() / (t_resid ** 2).sum()
```
### Comparison

The naive estimate is biased because treated households are systematically different (e.g., poorer, less educated).

The causal estimate corrects for this by removing the influence of confounders, leading to a more reliable estimate of the true effect.
Predict which estimate is larger, explain the omitted-variable issue with `caste_category`, and explain the cross-fitting problem.

### Interpretation

The naive estimate (Code A) is biased due to negative selection: households that participate in MGNREGA are systematically poorer, which pulls the estimate downward.

The adjusted approach (Code B) controls for confounders and provides a more credible estimate of the causal effect.

Caste is an important confounder because it influences both program participation and consumption. Omitting it would lead to biased estimates.

Additionally, using the same data for both model training and residual estimation can introduce overfitting bias. This is addressed through cross-fitting, where models are trained on separate folds of the data.

---

## 4. Estimating the Causal Effect

Now move from causal reasoning to actual estimation: partialling out, Double ML, diagnostics, and heterogeneity.

### OLS with Controls

We first estimate the effect using linear regression with observed confounders.

In [ ]:
def fit_linear_ols(df, outcome_col, treatment_col, control_cols):
    import statsmodels.api as sm

    # encode controls (handles categorical variables)
    _controls, _ = _encode_features(df, control_cols, add_fallback_column=True)

    X = _controls.copy()
    X[treatment_col] = df[treatment_col].values

    X = sm.add_constant(X)
    y = df[outcome_col].values

    model = sm.OLS(y, X).fit()
    return model

### Frisch-Waugh-Lovell (FWL) Approach

We partial out the effect of controls from both the outcome and treatment.

In [ ]:

def fwl_estimate(df, outcome_col, treatment_col, control_cols, n_folds=5, random_state=42):
    from sklearn.linear_model import LinearRegression

     # Step 0: encode controls
    _controls, _ = _encode_features(df, control_cols, add_fallback_column=True)

    # Step 1: outcome residual
    y = df[outcome_col].values
    model_y = LinearRegression().fit(_controls, y)
    y_resid = y - model_y.predict(_controls)

    # Step 2: treatment residual
    t = df[treatment_col].values
    model_t = LinearRegression().fit(_controls, t)
    t_resid = t - model_t.predict(_controls)

    # Step 3: residual on residual (NO intercept)
    coef = (y_resid * t_resid).sum() / (t_resid ** 2).sum()

    return {
        'fwl_coefficient': coef,
        'y_residuals': y_resid,
        't_residuals': t_resid
    }




### Double Machine Learning (DML)

We use machine learning models to flexibly control for confounders and estimate the treatment effect.

In [ ]:
def dml_estimate(df, outcome_col, treatment_col, confounder_cols, n_folds=5, random_state=42):
    from econml.dml import LinearDML

     # encode controls
    _controls, _ = _encode_features(df, confounder_cols, add_fallback_column=True)

    Y = df[outcome_col].values
    T = df[treatment_col].values
    W = _controls

    dml = LinearDML(cv=n_folds, random_state=random_state)
    dml.fit(Y, T, X=None, W=W)

    ate = dml.ate(X=None)
    ci = dml.ate_interval(X=None)

    return {
    'ate': float(ate),
    'ci_lower': float(ci[0]),
    'ci_upper': float(ci[1]),
    'se': float((ci[1] - ci[0]) / (2 * 1.96))
}

**Result:**  
After controlling for confounders, the estimated effect becomes positive, suggesting that the naive negative estimate was driven by selection bias.

## 5. Model Diagnostics

We evaluate:
- Model fit (R², AUC)
- Overlap in treatment assignment
- Stability across random seeds

These checks help assess whether the estimates are reliable.

In [ ]:
def dml_diagnostics(df, outcome_col, treatment_col, confounder_cols, random_state=42):
    from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
    from sklearn.metrics import r2_score, roc_auc_score
    import numpy as np

    # Encode controls
    _controls, _ = _encode_features(df, confounder_cols, add_fallback_column=True)

    Y = df[outcome_col].values
    T = df[treatment_col].values

    # -----------------------------
    # 1. Outcome model (R²)
    # -----------------------------
    model_y = GradientBoostingRegressor(random_state=random_state)
    model_y.fit(_controls, Y)
    y_pred = model_y.predict(_controls)
    outcome_r2 = r2_score(Y, y_pred)

    # -----------------------------
    # 2. Propensity model (AUC)
    # -----------------------------
    model_t = GradientBoostingClassifier(random_state=random_state)
    model_t.fit(_controls, T)
    t_pred = model_t.predict_proba(_controls)[:, 1]
    propensity_auc = roc_auc_score(T, t_pred)

    # -----------------------------
    # 3. Overlap check
    # -----------------------------
    overlap_violation_pct = np.mean((t_pred < 0.05) | (t_pred > 0.95)) * 100

    # -----------------------------
    # 4. Stability (3 seeds)
    # -----------------------------
    ates = []
    for seed in [0, 42, 99]:
        res = dml_estimate(df, outcome_col, treatment_col, confounder_cols, random_state=seed)
        ates.append(res['ate'])

    ate_range = max(ates) - min(ates)
    ate_stable = (ate_range / abs(np.mean(ates))) < 0.2

    # -----------------------------
    # Naive + OLS + DML
    # -----------------------------
    naive_estimate = float(
        df.loc[df[treatment_col]==1, outcome_col].mean() -
        df.loc[df[treatment_col]==0, outcome_col].mean()
    )

    ols_model = fit_linear_ols(df, outcome_col, treatment_col, confounder_cols)
    ols_estimate = float(ols_model.params[treatment_col])

    dml_res = dml_estimate(df, outcome_col, treatment_col, confounder_cols)
    dml_est = float(dml_res['ate'])

    # -----------------------------
    # Credibility
    # -----------------------------
    if outcome_r2 > 0.1 and propensity_auc > 0.6 and overlap_violation_pct < 20 and ate_stable:
        credibility = 'credible'
    elif overlap_violation_pct < 40:
        credibility = 'caution'
    else:
        credibility = 'not_credible'

    return {
        'outcome_r2': float(outcome_r2),
        'propensity_auc': float(propensity_auc),
        'overlap_violation_pct': float(overlap_violation_pct),
        'ate_stable': bool(ate_stable),
        'naive_estimate': float(naive_estimate),
        'ols_estimate': float(ols_estimate),
        'dml_estimate': float(dml_est),
        'credibility_assessment': credibility
    }


results = dml_diagnostics(df, 'consumption', 'mgnrega', CONFOUNDER_COLS)
results

## 6. Heterogeneous Treatment Effects

Using a causal forest, we estimate how the effect varies across households.

We examine differences across:
- Gender
- Caste groups
- Distance from town
- Land ownership

In [ ]:
def estimate_heterogeneous_effects(df, outcome_col, treatment_col, confounder_cols, effect_modifier_cols, random_state=42):
    from econml.dml import CausalForestDML
    import numpy as np

    # -----------------------------
    # Encode variables
    # -----------------------------
    W, _ = _encode_features(df, confounder_cols, add_fallback_column=True)
    X, _ = _encode_features(df, effect_modifier_cols, add_fallback_column=True)

    Y = df[outcome_col].values
    T = df[treatment_col].values

    # -----------------------------
    # Fit causal forest
    # -----------------------------
    cf = CausalForestDML(random_state=random_state)
    cf.fit(Y, T, X=X, W=W)

    cate = cf.effect(X)

    # -----------------------------
    # Mean CATE
    # -----------------------------
    mean_cate = float(np.mean(cate))

    # -----------------------------
    # Group comparisons
    # -----------------------------
    cate_by_group = {}

    cate_by_group['female_headed'] = float(np.mean(cate[df['head_female'] == 1]))
    cate_by_group['male_headed'] = float(np.mean(cate[df['head_female'] == 0]))

    for group in ['SC', 'ST', 'OBC', 'General']:
        cate_by_group[group] = float(np.mean(cate[df['caste_category'] == group]))

    cate_by_group['remote'] = float(np.mean(cate[df['distance_to_town_km'] > 30]))
    cate_by_group['near_town'] = float(np.mean(cate[df['distance_to_town_km'] <= 30]))

    cate_by_group['landless'] = float(np.mean(cate[df['land_acres'] < 1]))
    cate_by_group['landed'] = float(np.mean(cate[df['land_acres'] >= 1]))

    # -----------------------------
    # Top beneficiary quintile
    # -----------------------------
    threshold = np.percentile(cate, 80)
    top_mask = cate >= threshold

    top_profile = {
        'pct_female': float(np.mean(df.loc[top_mask, 'head_female'])),
        'pct_sc_st': float(np.mean(df.loc[top_mask, 'caste_category'].isin(['SC','ST']))),
        'mean_distance_km': float(np.mean(df.loc[top_mask, 'distance_to_town_km'])),
        'mean_land_acres': float(np.mean(df.loc[top_mask, 'land_acres']))
    }

    return {
        'cate_estimates': cate,
        'mean_cate': mean_cate,
        'cate_by_group': cate_by_group,
        'top_beneficiary_profile': top_profile
    }


results = estimate_heterogeneous_effects(
    df,
    'consumption',
    'mgnrega',
    CONFOUNDER_COLS,
    EFFECT_MODIFIER_COLS
)

print('Mean CATE:', round(q6_results['mean_cate'], 2))
print('Female vs male:', round(q6_results['cate_by_group']['female_headed'], 2), round(q6_results['cate_by_group']['male_headed'], 2))
print('Top-beneficiary profile:', q6_results['top_beneficiary_profile'])

**Insight:**  
Treatment effects are not uniform — some groups benefit significantly more than others.

## 7. Policy Targeting Under Budget Constraints

We simulate a scenario where only 60% of households can be treated.

We compare:
- Random allocation
- Targeted allocation using CATE estimates

## 8. Evaluating Common Causal Mistakes

We review common errors in applied ML:
- Controlling for post-treatment variables
- Confusing prediction with causation
- Ignoring overlap assumptions

In [ ]:
snippet_a = {
    'error_type': 'controls_for_mediator',
    'explanation': 'Skills gained and wage income are post-treatment variables, so controlling for them blocks part of the causal effect and introduces bias by conditioning on mediators.',
    'fix': 'Do not control for post-treatment variables. Only include pre-treatment confounders that affect both treatment and outcome.'
}

snippet_b = {
    'error_type': 'prediction_not_cate',
    'explanation': 'SHAP values from a prediction model explain outcome prediction, not causal treatment effects, so they do not identify who benefits from treatment.',
    'fix': 'Use causal methods like CATE estimation (e.g., CausalForestDML) to identify heterogeneous treatment effects instead of relying on prediction models.'
}

snippet_c = {
    'error_type': 'overlap_violation',
    'explanation': 'A precise estimate is unreliable if there is poor overlap, meaning some groups have little or no variation in treatment assignment, violating the positivity assumption.',
    'fix': 'Check overlap using propensity scores and restrict analysis to regions with sufficient overlap or adjust the sample to ensure valid comparisons.'
}

**Insight:**  
Targeting based on estimated treatment effects improves overall impact but may not align perfectly with equity goals.

---

###  The Targeting Recommendation

Suppose the budget is cut and only 60% of households can be covered. Use the estimated CATEs from Q6 to recommend a targeting rule.

When you compute the targeted group's female share, use the dataframe column `head_female`, but report it in the output dict under the key `pct_female`. In `equity_assessment`, explicitly compare the targeted group's female share and SC/ST share to the overall population shares.

In [ ]:
def targeting_recommendation(df, cate_estimates, budget_fraction=0.6):
    import numpy as np

    n = len(cate_estimates)
    k = int(budget_fraction * n)

    # -----------------------------
    # Select top CATE households
    # -----------------------------
    sorted_idx = np.argsort(cate_estimates)[::-1]
    targeted_indices = sorted_idx[:k]

    # -----------------------------
    # Total benefits
    # -----------------------------
    total_benefit_targeted = float(np.sum(cate_estimates[targeted_indices]))

    # random = average CATE * k
    total_benefit_random = float(np.mean(cate_estimates) * k)

    targeting_gain_pct = float(
        (total_benefit_targeted - total_benefit_random) / total_benefit_random * 100
    )

    # -----------------------------
    # Demographics of targeted group
    # -----------------------------
    target_df = df.iloc[targeted_indices]

    demographic_profile = {
        'pct_female': float(np.mean(target_df['head_female'])),
        'pct_sc_st': float(np.mean(target_df['caste_category'].isin(['SC','ST']))),
        'pct_remote': float(np.mean(target_df['distance_to_town_km'] > 30)),
        'pct_landless': float(np.mean(target_df['land_acres'] < 1))
    }

    # -----------------------------
    # Equity comparison
    # -----------------------------
    overall_pct_female = float(np.mean(df['head_female']))
    overall_pct_sc_st = float(np.mean(df['caste_category'].isin(['SC','ST'])))

    equity_assessment = (
        f"The targeted group has female share {demographic_profile['pct_female']:.2f} "
        f"compared to overall {overall_pct_female:.2f}, and SC/ST share "
        f"{demographic_profile['pct_sc_st']:.2f} compared to overall {overall_pct_sc_st:.2f}. "
        "This suggests the targeting rule prioritizes disadvantaged groups."
    )

    return {
        'targeted_indices': targeted_indices,
        'total_benefit_random': total_benefit_random,
        'total_benefit_targeted': total_benefit_targeted,
        'targeting_gain_pct': targeting_gain_pct,
        'demographic_profile': demographic_profile,
        'equity_assessment': equity_assessment
    }

results = targeting_recommendation(df, results['cate_estimates'], 0.6)


## 9. Robustness and Credibility

We test:
- Sensitivity to unobserved confounders
- Placebo outcomes (should not be affected by treatment)

These checks help validate the causal interpretation.

In [ ]:
def robustness_check(df, outcome_col, treatment_col, confounder_cols):

    # -----------------------------
    # 1. Sensitivity to unobservables
    # -----------------------------
    sensitivity_to_unobservables = (
        "A plausible unobserved confounder is household motivation or ability, "
        "which may affect both participation in MGNREGA and consumption levels."
    )

    # -----------------------------
    # 2. How strong would it need to be?
    # -----------------------------
    sensitivity_magnitude = "large"

    # -----------------------------
    # 3. Placebo test (use head_age)
    # -----------------------------
    placebo_result = dml_estimate(df, 'head_age', treatment_col, confounder_cols)
    placebo_test = float(placebo_result['ate'])

    placebo_passes = abs(placebo_test) < 2.0

    # -----------------------------
    # 4. Overall credibility
    # -----------------------------
    if placebo_passes:
        overall_credibility = "high"
    else:
        overall_credibility = "medium"

    return {
        'sensitivity_to_unobservables': sensitivity_to_unobservables,
        'sensitivity_magnitude': sensitivity_magnitude,
        'placebo_test': placebo_test,
        'placebo_passes': placebo_passes,
        'overall_credibility': overall_credibility
    }

q9_results = robustness_check(df, 'consumption', 'mgnrega', CONFOUNDER_COLS)


## Conclusion

- The naive estimate misleadingly suggests a negative effect
- After controlling for confounders, MGNREGA increases consumption
- Selection bias explains the initial paradox
- Treatment effects are heterogeneous
- Targeting improves efficiency but raises equity considerations

This highlights the importance of causal inference in policy evaluation.